In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from time import sleep
import json


In [61]:
browser = webdriver.Firefox()
browser.get('https://www.olx.in/en-in')
sleep(2)


In [62]:
location_box = browser.find_element(By.XPATH,'//input[@placeholder="Search city, area or locality"]')
location_box.clear()  
location_box.send_keys("Bhopal")
location_box.send_keys(Keys.ENTER)
sleep(10)

In [63]:
location_container = browser.find_element(By.XPATH,"//div[@data-aut-id='locationItem']//span[contains(., 'Bhopal')]")
location_container.click()
sleep(5)

In [ ]:
search_box = browser.find_element(By.XPATH, "//input[@data-aut-id='searchBox']")
search_box.send_keys("house rent")
search_box.send_keys(Keys.ENTER)

In [68]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# wait until listings are present
WebDriverWait(browser, 10).until(
    EC.presence_of_element_located((By.XPATH, '//a[.//span[@data-aut-id="itemTitle"]]'))
)

products = browser.find_elements(
    By.XPATH,
    '//a[.//figure[@data-aut-id="itemImage"]]'
)

for product in products:
    title = product.find_element(By.XPATH, './/span[@data-aut-id="itemTitle"]').text
    price = product.find_element(By.XPATH, './/span[@data-aut-id="itemPrice"]').text
    details = product.find_element(By.XPATH, './/span[@data-aut-id="itemDetails"]').text
    location = product.find_element(By.XPATH, './/span[@data-aut-id="item-location"]').text
    link = product.get_attribute("href")


    print(title, price, details, location, link)

Premium independent property for batchelors ₹ 9,000 2 BHK - 1 Bathroom - 1200 sqft ASHOKA GARDEN, BHOPAL https://www.olx.in/en-in/item/for-rent-houses-apartments-c1723-2-bhk-apartments-1200-sq-ft-in-ashoka-garden-bhopal-iid-1834676475
Beautiful small 1 BRK, semi furnished, very near 2 excellence college ₹ 8,000 1 BHK - 1 Bathroom - 500 sqft CHUNABHATTI, BHOPAL https://www.olx.in/en-in/item/for-rent-houses-apartments-c1723-1-bhk-apartments-500-sq-ft-in-chunabhatti-bhopal-iid-1833088250
3bhk furnished independent house in govind garden near roshan hospital ₹ 25,000 3 BHK - 3 Bathroom - 1500 sqft GOVINDPURA, BHOPAL https://www.olx.in/en-in/item/for-rent-houses-apartments-c1723-3-bhk-houses-villas-1500-sq-ft-in-govindpura-bhopal-iid-1835422371
Rent for 3bhk Duplex semi furnished covered campus gulmohar. ₹ 25,000 3 BHK - 3 Bathroom - 1800 sqft ROHIT NAGAR, BHOPAL https://www.olx.in/en-in/item/for-rent-houses-apartments-c1723-3-bhk-houses-villas-1800-sq-ft-in-rohit-nagar-bhopal-iid-183105040

NoSuchElementException: Message: Unable to locate element: .//span[@data-aut-id="itemDetails"]; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:555:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:136:16


In [69]:
import json
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

def safe_text(parent, xpath):
    try:
        return parent.find_element(By.XPATH, xpath).text.strip()
    except NoSuchElementException:
        return None

# wait until listings are present
WebDriverWait(browser, 10).until(
    EC.presence_of_element_located((By.XPATH, '//a[.//span[@data-aut-id="itemTitle"]]'))
)

products = browser.find_elements(
    By.XPATH,
    '//a[.//figure[@data-aut-id="itemImage"]]'
)

data = []

for product in products:
    item = {
        "title": safe_text(product, './/span[@data-aut-id="itemTitle"]'),
        "price": safe_text(product, './/span[@data-aut-id="itemPrice"]'),
        "details": safe_text(product, './/span[@data-aut-id="itemDetails"]'),  # this was crashing
        "location": safe_text(product, './/span[@data-aut-id="item-location"]'),
        "link": product.get_attribute("href")
    }
    data.append(item)

with open("listings.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data)} listings to listings.json")

Saved 40 listings to listings.json


In [70]:
import time
import json
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

wait = WebDriverWait(browser, 10)

# Step 1: Keep clicking "Load more" until it disappears
while True:
    try:
        load_more_btn = wait.until(
            EC.element_to_be_clickable((By.XPATH, '//button[@data-aut-id="btnLoadMore"]'))
        )

        browser.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
        time.sleep(0.5)  # small pause for smooth scrolling

        browser.execute_script("arguments[0].click();", load_more_btn)
        time.sleep(2)  # wait for new items to load

    except TimeoutException:
        print("No more 'Load more' button. All listings loaded.")
        break

    except (StaleElementReferenceException, NoSuchElementException):
        # DOM refreshed, retry loop
        time.sleep(1)
        continue

# Step 2: Now scrape everything
wait.until(
    EC.presence_of_element_located((By.XPATH, '//a[.//span[@data-aut-id="itemTitle"]]'))
)

products = browser.find_elements(By.XPATH, '//a[.//figure[@data-aut-id="itemImage"]]')

from selenium.common.exceptions import NoSuchElementException

def safe_text(parent, xpath):
    try:
        return parent.find_element(By.XPATH, xpath).text.strip()
    except NoSuchElementException:
        return None

data = []

for product in products:
    title = safe_text(product, './/span[@data-aut-id="itemTitle"]')
    link = product.get_attribute("href")

    if not title or not link:
        continue

    data.append({
        "title": title,
        "price": safe_text(product, './/span[@data-aut-id="itemPrice"]'),
        "details": safe_text(product, './/span[@data-aut-id="itemDetails"]'),
        "location": safe_text(product, './/span[@data-aut-id="item-location"]'),
        "link": link
    })

with open("listings.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data)} listings to listings.json")

No more 'Load more' button. All listings loaded.
Saved 1080 listings to listings.json
